# Ingest DataFrame Traces into Fiddler

This notebook demonstrates how to load event data from a CSV file and ingest it
into a Fiddler LLM application as OpenTelemetry traces using .

## Prerequisites

- A Fiddler instance with an LLM application model
- The application UUID, agent ID, and agent name from your Fiddler setup
- A CSV file with trace/span data to ingest

In [ ]:
%pip install -q "fiddler-utils[otel] @ git+https://github.com/fiddler-labs/fiddler-utils.git@v1.1.1"

## Configuration

In [ ]:
import os

URL = os.environ.get("FIDDLER_URL", "https://your-org.cloud.fiddler.ai")
API_KEY = os.environ.get("FIDDLER_API_KEY", "YOUR_FIDDLER_API_KEY")
APP_UUID = os.environ.get("FIDDLER_APP_UUID", "your-app-uuid")

AGENT_ID = os.environ.get("FIDDLER_AGENT_ID", "your-agent-id")
AGENT_NAME = os.environ.get("FIDDLER_AGENT_NAME", "your-agent-name")

PATH_TO_CSV_FILE = os.environ.get("PATH_TO_CSV_FILE", "/path/to/your/traces.csv")

## Initialize the Fiddler client

In [ ]:
import logging
import sys
import pandas as pd
from fiddler.libs.http_client import RequestClient
from fiddler_utils.assets.ingestion import log_pandas_traces

logging.basicConfig(
    level=logging.INFO,
    stream=sys.stdout,
    force=True
)

# 1. Create your RequestClient with Fiddler credentials
client = RequestClient(
    base_url=URL,
    headers={
        'Authorization': f'Bearer {API_KEY}',
        'Fiddler-Application-Id': APP_UUID
    }
)

## Load in a CSV file as a pandas DataFrame

In [ ]:
df = pd.read_csv(PATH_TO_CSV_FILE)
df

## Map DataFrame columns to Fiddler semantic conventions

In [ ]:
column_mapping = {
    'question': 'gen_ai.llm.input.user',
    'response': 'gen_ai.llm.output',
    'session_id': 'gen_ai.conversation.id',
    'start_time': 'timestamp',
    'end_time': 'timestamp'
}

## Set static attributes that apply to ALL spans

In [ ]:
static_attrs = {
    'gen_ai.agent.id': AGENT_ID,
    'gen_ai.agent.name': AGENT_NAME,
    'fiddler.span.type': 'llm',
}

## Ingest DataFrame rows as OTEL traces

In [ ]:
log_pandas_traces(
    df=df,
    fiddler_client=client,
    column_mapping=column_mapping,
    static_attributes=static_attrs
)